# GLAB 925.5.1 — Exploring Airbnb Market Trends Using AI Assistance
**Student:** Oscar Eustate  
**Scope:** Descriptive analysis of the supplied 2019 New York City Airbnb listings, with a focus on private rooms.

## How to run
Open this notebook in Google Colab and select **Runtime → Run all**. The next cell downloads the three assignment files if they are absent. If Google blocks a download, upload all three original files when prompted. Locally, place the files alongside this notebook and install `pandas numpy matplotlib openpyxl` first.

Run all cells, review the generated findings, and save/download the executed `.ipynb` before submitting it in Canvas. This notebook was authored with AI assistance; no source-data results are prefilled. All tables, figures, and numerical interpretations below are calculated when run.

## Sources and relationship
- [airbnb_price.csv](https://drive.google.com/file/d/1r9D6ElQHM4jL9ai7c-hjFpEqDwv0eHCr/view?usp=sharing): nightly USD price and location.
- [airbnb_room_type.xlsx](https://docs.google.com/spreadsheets/d/1wzpjNQiCtCsYZST4O7lNUB6KWe6AO01H/edit?usp=sharing): description and room type.
- [airbnb_last_review.tsv](https://drive.google.com/file/d/13D-XY_lY57XZaHirdlCb-dhOphQN9TWj/view?usp=sharing): host and last-review date.

The common key is `listing_id`. A one-to-one join is appropriate only after checking that IDs are unique and nonmissing in each file. The analysis uses the intersection of IDs and reports unmatched IDs before joining.


In [ ]:
from pathlib import Path
from urllib.request import Request, urlopen
import importlib.util
import sys
import subprocess

# Colab normally includes these; install only missing packages in Colab.
packages = ['pandas', 'numpy', 'matplotlib', 'openpyxl']
missing_packages = [p for p in packages if importlib.util.find_spec(p) is None]
if missing_packages:
    if not (importlib.util.find_spec('google') and importlib.util.find_spec('google.colab')):
        raise ImportError('Install these packages first: ' + ' '.join(missing_packages))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
plt.style.use('seaborn-v0_8-whitegrid')

source_ids = {
    'airbnb_price.csv': '1r9D6ElQHM4jL9ai7c-hjFpEqDwv0eHCr',
    'airbnb_room_type.xlsx': '1wzpjNQiCtCsYZST4O7lNUB6KWe6AO01H',
    'airbnb_last_review.tsv': '13D-XY_lY57XZaHirdlCb-dhOphQN9TWj',
}
for filename, file_id in source_ids.items():
    if Path(filename).exists():
        continue
    try:
        url = f'https://drive.google.com/uc?export=download&id={file_id}'
        request = Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urlopen(request, timeout=45) as response:
            payload = response.read()
        beginning = payload[:1000].lower()
        if b'<html' in beginning or b'<!doctype html' in beginning:
            raise ValueError('Google returned a web page instead of the source file.')
        if filename.endswith('.xlsx') and not payload.startswith(b'PK'):
            raise ValueError('The response is not an XLSX file.')
        Path(filename).write_bytes(payload)
        print('Downloaded:', filename)
    except Exception as exc:
        print(f'Could not download {filename}: {exc}')

missing_files = [f for f in source_ids if not Path(f).exists()]
if missing_files and importlib.util.find_spec('google') and importlib.util.find_spec('google.colab'):
    from google.colab import files
    print('Upload the original assignment files:', ', '.join(missing_files))
    files.upload()
missing_files = [f for f in source_ids if not Path(f).exists()]
if missing_files:
    raise FileNotFoundError('Place these original files beside the notebook: ' + ', '.join(missing_files))


## 1. Import, inspect, and join the three files


In [ ]:
airbnb_price = pd.read_csv('airbnb_price.csv')
airbnb_room_type = pd.read_excel('airbnb_room_type.xlsx', engine='openpyxl')
airbnb_last_review = pd.read_csv('airbnb_last_review.tsv', sep='\t')
sources = {'prices': airbnb_price, 'room types': airbnb_room_type, 'reviews': airbnb_last_review}
required = {
    'prices': {'listing_id', 'price', 'nbhood_full'},
    'room types': {'listing_id', 'description', 'room_type'},
    'reviews': {'listing_id', 'host_name', 'last_review'},
}
audit = []
for name, frame in sources.items():
    assert required[name].issubset(frame.columns), f'Missing columns in {name}'
    ids = pd.to_numeric(frame['listing_id'], errors='raise')
    assert ids.notna().all() and np.isfinite(ids).all() and (ids % 1 == 0).all(), 'Invalid ID'
    frame['listing_id'] = ids.astype('int64')
    audit.append({'source': name, 'rows': len(frame), 'unique_ids': frame.listing_id.nunique(),
                  'duplicate_ids': int(frame.listing_id.duplicated().sum())})
    assert frame.listing_id.is_unique, f'Duplicate IDs in {name}; investigate before merging.'
    display(Markdown(f'**{name.title()} — first five rows**'))
    display(frame.head())
    display(frame.isna().sum().rename('source_missing_values').to_frame())

display(pd.DataFrame(audit))
id_sets = {name: set(frame.listing_id) for name, frame in sources.items()}
common_ids = set.intersection(*id_sets.values())
display(pd.DataFrame([
    {'source': name, 'ids_absent_from_at_least_one_other_file': len(ids - common_ids)}
    for name, ids in id_sets.items()
]))
listings = airbnb_price.merge(airbnb_room_type, on='listing_id', how='inner', validate='one_to_one')
listings = listings.merge(airbnb_last_review, on='listing_id', how='inner', validate='one_to_one')
assert len(listings) == len(common_ids) > 0
print(f'Joined table: {len(listings):,} listings; one row per listing.')
print('Column names:', listings.columns.tolist())
display(listings.head())


## 2. Clean prices, room types, locations, and review dates
Preserve raw values for auditing. Prices may contain dollar signs, commas, or a trailing “dollars.” Failed conversions remain missing. Zero or negative prices are flagged and excluded from pricing summaries; those listings remain in inventory and review counts. Missing host names or descriptions do not require dropping otherwise usable records. No prices or dates are imputed.


In [ ]:
raw_missingness = listings.isna().sum()
listings['price_raw'] = listings['price']
price_text = (listings['price'].astype('string').str.strip()
              .str.replace(r'(?i)\s*(?:usd|dollars?)\s*', '', regex=True)
              .str.replace(r'[$,]', '', regex=True).str.strip())
listings['price'] = pd.to_numeric(price_text, errors='coerce')
listings['room_type_raw'] = listings['room_type']
listings['room_type'] = listings['room_type'].astype('string').str.strip().str.lower()
listings['room_type'] = listings['room_type'].replace({'entire home/apartment': 'entire home/apt'})
allowed = {'private room', 'shared room', 'entire home/apt'}
unexpected = set(listings.room_type.dropna().unique()) - allowed
assert not unexpected, f'Investigate unexpected room types: {unexpected}'

listings['last_review_date'] = pd.to_datetime(
    listings['last_review'].astype('string').str.strip(), format='%B %d %Y', errors='coerce')
# The first comma separates borough from neighborhood; keep the rest together.
location = listings['nbhood_full'].astype('string').str.split(',', n=1, expand=True)
listings['borough'] = location[0].str.strip().replace('', pd.NA)
listings['neighborhood'] = location.reindex(columns=[1])[1].str.strip().replace('', pd.NA)
price_failures = listings.price_raw.notna() & listings.price.isna()
date_failures = listings.last_review.notna() & listings.last_review_date.isna()
invalid_price = listings.price.notna() & ((listings.price <= 0) | ~np.isfinite(listings.price))
print('Price conversion failures:', int(price_failures.sum()))
print('Date conversion failures:', int(date_failures.sum()))
print('Nonpositive or nonfinite prices:', int(invalid_price.sum()))
for mask, columns in [(price_failures, ['listing_id', 'price_raw']),
                      (date_failures, ['listing_id', 'last_review']),
                      (invalid_price, ['listing_id', 'price'])]:
    if mask.any():
        display(listings.loc[mask, columns].head(10))

display(pd.DataFrame({'raw_missing': raw_missingness,
                      'clean_missing': listings[raw_missingness.index].isna().sum()}))
display(listings[['price', 'room_type', 'last_review_date', 'borough', 'neighborhood']]
        .isna().agg(['sum', 'mean']).T.rename(columns={'sum': 'missing_count', 'mean': 'missing_fraction'}))
priced = listings.loc[listings.price.notna() & ~invalid_price].copy()
assert len(priced) > 0, 'No usable prices.'
first_reviewed = listings.last_review_date.min()
last_reviewed = listings.last_review_date.max()
assert pd.notna(last_reviewed), 'No valid review dates; inspect the source format.'
private_room_count = int(listings.room_type.eq('private room').sum())
avg_price = priced.price.mean()
print('Earliest last-review date:', first_reviewed.date())
print('Most recent last-review date:', last_reviewed.date())
print(f'Private-room listings: {private_room_count:,}')
print(f'Average nightly price (positive finite prices): ${avg_price:,.2f}')


## 3. Overall pricing, tails, and IQR outliers


In [ ]:
price_summary = priced.price.describe(percentiles=[.01, .25, .5, .75, .9, .95, .99])
display(price_summary.to_frame('nightly_price_usd'))
q1, median, q3 = priced.price.quantile([.25, .5, .75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
priced['iqr_outlier'] = (priced.price < lower) | (priced.price > upper)
print(f'Q1=${q1:,.2f}; median=${median:,.2f}; Q3=${q3:,.2f}; IQR=${iqr:,.2f}')
print(f'Overall 1.5 × IQR fences: ${lower:,.2f} to ${upper:,.2f}')
print(f'Flagged listings: {priced.iqr_outlier.sum():,} ({priced.iqr_outlier.mean():.1%})')
display(priced.loc[priced.iqr_outlier, ['listing_id', 'room_type', 'borough', 'price']]
        .sort_values('price', ascending=False).head(10))
# Sensitivity only: the primary summaries retain all valid prices.
print(f'Mean retaining outliers: ${priced.price.mean():,.2f}')
print(f'Mean excluding IQR flags: ${priced.loc[~priced.iqr_outlier, "price"].mean():,.2f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
axes[0].hist(priced.price, bins=60, color='#197a8c')
axes[0].set(title='Full price distribution', xlabel='Nightly price (USD)', ylabel='Listings')
axes[1].hist(np.log10(priced.price), bins=50, color='#b66a36')
axes[1].set(title='Price distribution on a log scale', xlabel='log10(nightly price in USD)', ylabel='Listings')
plt.show()
display(Markdown(f"""**Pricing interpretation.** The mean is **${avg_price:,.2f}** and the median is
**${median:,.2f}**. The middle 50% span **${q1:,.2f}–${q3:,.2f}** (IQR **${iqr:,.2f}**).
The 95th and 99th percentiles are **${priced.price.quantile(.95):,.2f}** and
**${priced.price.quantile(.99):,.2f}**. Compare the mean, median, and histograms to assess the upper tail.
IQR flags identify unusual observations to investigate; they do not prove data errors.
Do not automatically cap a plausible luxury listing merely because it is expensive."""))


## 4. Room-type mix, prices, and price bands


In [ ]:
room_order = ['entire home/apt', 'private room', 'shared room']
mix = listings.room_type.fillna('unknown').value_counts().rename_axis('room_type').to_frame('listings')
mix['share_pct'] = 100 * mix.listings / len(listings)
display(mix)
room_summary = priced.groupby('room_type', dropna=False).price.agg(
    count='count', mean='mean', median='median', min='min', max='max',
    q1=lambda s: s.quantile(.25), q3=lambda s: s.quantile(.75))
room_summary['iqr'] = room_summary.q3 - room_summary.q1
display(room_summary.sort_values('median', ascending=False))

# Within-type fences avoid treating differences between room types as anomalies.
segment_q1 = priced.groupby('room_type').price.transform(lambda s: s.quantile(.25))
segment_q3 = priced.groupby('room_type').price.transform(lambda s: s.quantile(.75))
segment_iqr = segment_q3 - segment_q1
priced['within_type_outlier'] = ((priced.price < segment_q1 - 1.5 * segment_iqr)
                               | (priced.price > segment_q3 + 1.5 * segment_iqr))
display(priced.groupby('room_type').within_type_outlier.agg(['sum', 'mean'])
        .rename(columns={'sum': 'flagged_listings', 'mean': 'flagged_fraction'}))

# Analyst-defined bands, not official market categories. Right endpoint is excluded.
bands = ['Under $50', '$50–99', '$100–199', '$200–299', '$300+']
priced['price_band'] = pd.cut(priced.price, [0, 50, 100, 200, 300, np.inf],
                              right=False, labels=bands)
band_counts = pd.crosstab(priced.price_band, priced.room_type.fillna('unknown'), dropna=False)
band_shares = band_counts.div(band_counts.sum(axis=1).replace(0, np.nan), axis=0) * 100
display(Markdown('**Counts by price band and room type**'))
display(band_counts)
display(Markdown('**Room-type composition within each price band (%)**'))
display(band_shares)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), constrained_layout=True)
mix.share_pct.plot.bar(ax=axes[0], color='#197a8c', rot=25)
axes[0].set(title='Inventory mix', ylabel='Share of all joined listings (%)', xlabel='Room type')
present = [r for r in room_order if priced.room_type.eq(r).any()]
axes[1].boxplot([priced.loc[priced.room_type.eq(r), 'price'] for r in present], showfliers=True)
axes[1].set_xticks(range(1, len(present) + 1), present)
axes[1].set_yscale('log')
axes[1].tick_params(axis='x', labelrotation=25)
axes[1].set(title='Prices by room type (all valid prices)', ylabel='Nightly USD (log scale)')
band_shares.plot.bar(stacked=True, ax=axes[2], rot=25, colormap='Set2')
axes[2].set(title='Room mix within price bands', ylabel='Share within band (%)', xlabel='Price band')
axes[2].legend(title='Room type', fontsize=8)
plt.show()

ranked_rooms = room_summary.dropna(subset=['median']).sort_values('median')
display(Markdown(f"""**Room-type interpretation.** By median nightly price,
**{ranked_rooms.index[-1]}** is most expensive (**${ranked_rooms.iloc[-1]['median']:,.2f}**),
and **{ranked_rooms.index[0]}** is least expensive (**${ranked_rooms.iloc[0]['median']:,.2f}**).
There are **{private_room_count:,} private rooms**, representing **{private_room_count / len(listings):.1%}**
of the joined inventory. Price-band percentages use the listings within each band as their denominator."""))
for band, row in band_shares.iterrows():
    if row.notna().any():
        print(f'{band}: largest segment = {row.idxmax()} ({row.max():.1f}% of that band).')


## 5. Location and private-room positioning
Borough and neighborhood are available in `nbhood_full`. Compare medians within room type as well as across boroughs, because borough-wide differences may partly reflect room-type mix. Small groups provide less stable benchmarks; the full table retains their counts.


In [ ]:
borough_summary = priced.groupby('borough', dropna=False).price.agg(['count', 'mean', 'median'])
display(borough_summary.sort_values('median', ascending=False))
display(priced.pivot_table(index='borough', columns='room_type', values='price', aggfunc='median'))
private = priced.loc[priced.room_type.eq('private room')].copy()
assert len(private) > 0, 'No private rooms have usable prices.'
private_borough = private.groupby('borough', dropna=False).price.agg(['count', 'mean', 'median'])
display(Markdown('**Private-room pricing by borough**'))
display(private_borough.sort_values('median', ascending=False))
private_neighborhood = private.groupby(['borough', 'neighborhood'], dropna=False).price.agg(['count', 'median'])
display(Markdown('**Highest private-room neighborhood medians among groups with at least 10 priced listings**'))
display(private_neighborhood.loc[private_neighborhood['count'] >= 10]
        .sort_values('median', ascending=False).head(10))
private_borough.sort_values('median')['median'].plot.barh(figsize=(8, 4), color='#197a8c')
plt.title('Private-room median nightly price by borough')
plt.xlabel('Median nightly price (USD)')
plt.tight_layout()
plt.show()
private_q1, private_median, private_q3 = private.price.quantile([.25, .5, .75])
display(Markdown(f"""**Private-room interpretation.** Among the **{len(private):,}** private rooms with
usable prices, the median is **${private_median:,.2f}**, the mean is **${private.price.mean():,.2f}**,
and the middle 50% range is **${private_q1:,.2f}–${private_q3:,.2f}**.
Use these as historical comparison points and refine them with the borough and neighborhood tables.
The 10-listing threshold is a reporting choice, not proof of statistical reliability.
These descriptive differences do not isolate the causal effect of location or property quality."""))


## 6. Review recency and monthly last-review activity
Each listing contributes only its **last** review date. Monthly counts measure how many listings have their latest recorded review in that month, rather than the total number of reviews or bookings. Use the latest observed date as a historical reference instead of comparing 2019 data with today's date.


In [ ]:
valid_dates = listings.last_review_date.dropna()
month_index = pd.period_range(valid_dates.min().to_period('M'), valid_dates.max().to_period('M'), freq='M')
monthly = valid_dates.dt.to_period('M').value_counts().reindex(month_index, fill_value=0).sort_index()
monthly.index.name = 'last_review_month'
display(monthly.rename('listings_with_last_review').to_frame())
listings['days_since_last_review'] = (last_reviewed - listings.last_review_date).dt.days
display(listings.days_since_last_review.describe().to_frame('days_before_latest_observed_review'))
recent_count = int(listings.days_since_last_review.between(0, 30).sum())
print(f'Latest observed review: {last_reviewed.date()}')
print(f'Listings within 30 days of that date (inclusive): {recent_count:,} / {len(valid_dates):,} dated listings')

monthly.plot.bar(figsize=(10, 4), color='#197a8c', rot=35)
plt.title('Listings by month of their latest recorded review')
plt.xlabel('Last-review month')
plt.ylabel('Listings (not total reviews or bookings)')
plt.tight_layout()
plt.show()
peak_months = ', '.join(str(m) for m in monthly.index[monthly.eq(monthly.max())])
display(Markdown(f"""**Review interpretation.** The last-review dates range from
**{first_reviewed.date()}** to **{last_reviewed.date()}**. The largest monthly last-review count is
**{monthly.max():,}**, in **{peak_months}**. **{recent_count / len(valid_dates):.1%}** of listings with
a valid date have their latest review within 30 days of the latest observed date.
The first and last months may have incomplete coverage. A recent-month concentration is also expected
when recording only each listing's latest review. This snapshot cannot establish seasonal demand peaks,
or whether demand is heating or cooling; that would require complete review or booking histories across comparable periods."""))


## 7. Correlations and verification
Price and review recency are the only two meaningful continuous measures supplied or directly derived here. Listing ID is an identifier, and categorical room types and boroughs have no meaningful arbitrary numeric order. Compare those categories using the group summaries instead. Correlations below are exploratory associations and do not measure demand or causality.


In [ ]:
corr_data = priced[['listing_id', 'price']].merge(
    listings[['listing_id', 'days_since_last_review']], on='listing_id', validate='one_to_one')
paired = corr_data[['price', 'days_since_last_review']].dropna()
print(f'Complete price/recency pairs: {len(paired):,}')
display(paired.corr(method='pearson'))
# Spearman is Pearson correlation of average ranks (avoids an extra SciPy dependency).
display(paired.rank(method='average').corr(method='pearson'))
pearson_r = paired.price.corr(paired.days_since_last_review)
spearman_r = paired.price.rank().corr(paired.days_since_last_review.rank())
display(Markdown(f"""**Correlation interpretation.** Pearson r = **{pearson_r:.3f}** and Spearman
rho = **{spearman_r:.3f}** for price versus days before the latest observed review.
A positive value means higher prices are associated with older last reviews; a negative value means
higher prices are associated with more recent last reviews. Values near zero indicate little association
on the respective scale. Pearson is sensitive to the price tail; Spearman compares ranks.
Neither controls for location, room type, or listing characteristics."""))

# Independent checks catch join expansion, missing categories, and lost observations.
assert listings.listing_id.is_unique
assert set(listings.listing_id) == common_ids
assert mix.listings.sum() == len(listings)
assert np.isclose(mix.share_pct.sum(), 100)
assert band_counts.to_numpy().sum() == len(priced)
assert monthly.sum() == listings.last_review_date.notna().sum()
assert private_room_count == int(np.count_nonzero(listings.room_type.fillna('').to_numpy() == 'private room'))
assert np.isclose(avg_price, np.mean(priced.price.to_numpy()))
assert first_reviewed <= last_reviewed
print('Verification passed: joins, inventory shares, price bands, monthly counts, and mean reconcile.')


## 8. Consultant findings and AI-assistance reflection


In [ ]:
private_location_rank = private_borough.dropna(subset=['median']).sort_values('median')
location_text = ''
if len(private_location_rank):
    location_text = (f'The highest observed private-room borough median is '
                     f'**{private_location_rank.index[-1]}** '
                     f'(**${private_location_rank.iloc[-1]["median"]:,.2f}**); '
                     f'compare its sample count and neighborhood mix before drawing conclusions.')
display(Markdown(f"""
### Findings for the real estate start-up
- **Inventory:** Private rooms account for **{private_room_count:,} listings ({private_room_count / len(listings):.1%})**.
- **Pricing benchmark:** The historical private-room median is **${private_median:,.2f}** per night;
  its middle 50% span **${private_q1:,.2f}–${private_q3:,.2f}**. Medians and IQRs are useful comparison
  points when expensive outliers influence the average.
- **Location:** {location_text}
- **Unusual listings:** Review the **{priced.iqr_outlier.sum():,}** overall IQR flags and the within-type
  flags for data errors or distinctive amenities. Retain valid extremes; report any capping as a separate sensitivity analysis.
- **Activity:** The monthly peak is **{monthly.max():,}** latest-review dates in **{peak_months}**.
  This describes the snapshot's review recency, not a demonstrated peak in demand.
- **Next step:** Compare private rooms within the same neighborhood and collect occupancy, complete review
  histories, fees, amenities, and operating costs before evaluating revenue or profitability.

### AI-assistance record
ChatGPT/Codex assisted with drafting the imports, validated joins, data cleaning, summary statistics,
IQR flags, charts, and interpretation text. The work addresses the lab prompts: relationships between files,
joining on listing ID, listing columns, earliest/latest reviews, normalized private-room counts, and mean prices.
Verification code checks one row per ID, join membership, price-band and review-count reconciliation,
inventory shares, and an independent NumPy mean calculation. Inspect the tables and charts after execution
to verify the generated interpretation against the data. This records AI's contribution without claiming
that the student has already performed a personal review.

### Limitations
The data describe the supplied 2019 sample, not the current NYC market. Inner joins restrict the analysis
to common IDs; source exclusions are shown above. Missing prices and nonpositive/nonfinite prices are
excluded only from price analyses, and missing dates only from review analyses. Advertised prices are
not realized revenue and may exclude fees. Room-type and location comparisons are descriptive;
the dataset lacks controls needed to establish causes, occupancy, demand trends, or profitability.
"""))
